# SQL Business Analysis

## Purpose

This notebook uses DuckDB and SQL to calculate the main business metrics for the project.

The initial analysis focuses on:

- order status;
- customer review scores;
- delivery delays;
- differences in customer satisfaction between on-time and delayed orders.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

orders_file = project_root / "data" / "raw" / "olist_orders_dataset.csv"
reviews_file = project_root / "data" / "raw" / "olist_order_reviews_dataset.csv"

print("Orders file found:", orders_file.exists())
print("Reviews file found:", reviews_file.exists())

Orders file found: True
Reviews file found: True


In [2]:
order_status_summary = duckdb.execute(
    """
    SELECT
        order_status,
        COUNT(*) AS order_count
    FROM read_csv_auto(?)
    GROUP BY order_status
    ORDER BY order_count DESC
    """,
    [orders_file.as_posix()]
).df()

order_status_summary

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [3]:
order_status_metrics = duckdb.execute(
    """
    WITH orders AS (
        SELECT *
        FROM read_csv_auto(?)
    )
    
    SELECT
        order_status,
        COUNT(*) AS order_count,
        ROUND(
            100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders),
            2
        ) AS order_percent
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC
    """,
    [orders_file.as_posix()]
).df()

order_status_metrics

,order_status,order_count,order_percent
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


## 1. Order status

A total of 99,441 orders are recorded in the dataset. Approximately 97.02% were marked as delivered, while the remaining orders were shipped, cancelled, unavailable or still at an earlier processing stage.

The delivery-delay analysis will therefore focus on delivered orders with valid actual and estimated delivery dates.

In [4]:
review_score_metrics = duckdb.execute(
    """
    WITH reviews AS (
        SELECT *
        FROM read_csv_auto(?)
    )
    
    SELECT
        review_score,
        COUNT(*) AS review_count,
        ROUND(
            100.0 * COUNT(*) / (SELECT COUNT(*) FROM reviews),
            2
        ) AS review_percent
    FROM reviews
    GROUP BY review_score
    ORDER BY review_score
    """,
    [reviews_file.as_posix()]
).df()

review_score_metrics

,review_score,review_count,review_percent
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


## 2. Customer review scores

Customer ratings are concentrated at the upper end of the scale. Five-star reviews account for 57.78% of all review records, while four- and five-star reviews together account for 77.07%.

Low ratings of one or two stars represent 14.69% of review records. This distribution suggests that most customers report positive experiences, but a meaningful minority are dissatisfied.

These figures are preliminary and may be revised after duplicate review records are examined.

In [5]:
satisfaction_group_metrics = duckdb.execute(
    """
    WITH reviews AS (
        SELECT *
        FROM read_csv_auto(?)
    ),
    
    classified_reviews AS (
        SELECT
            review_score,
            CASE
                WHEN review_score IN (1, 2) THEN 'Low rating'
                WHEN review_score = 3 THEN 'Neutral rating'
                WHEN review_score IN (4, 5) THEN 'High rating'
            END AS satisfaction_group
        FROM reviews
    )
    
    SELECT
        satisfaction_group,
        COUNT(*) AS review_count,
        ROUND(
            100.0 * COUNT(*) /
            (SELECT COUNT(*) FROM classified_reviews),
            2
        ) AS review_percent
    FROM classified_reviews
    GROUP BY satisfaction_group
    ORDER BY review_percent DESC
    """,
    [reviews_file.as_posix()]
).df()

satisfaction_group_metrics

,satisfaction_group,review_count,review_percent
0,High rating,76470,77.07
1,Low rating,14575,14.69
2,Neutral rating,8179,8.24


In [6]:
delivery_status_metrics = duckdb.execute(
    """
    WITH delivered_orders AS (
        SELECT
            order_id,
            DATE_DIFF(
                'day',
                CAST(order_estimated_delivery_date AS DATE),
                CAST(order_delivered_customer_date AS DATE)
            ) AS delivery_delay_days
        FROM read_csv_auto(?)
        WHERE order_status = 'delivered'
          AND order_delivered_customer_date IS NOT NULL
          AND order_estimated_delivery_date IS NOT NULL
    ),
    
    classified_orders AS (
        SELECT
            order_id,
            delivery_delay_days,
            CASE
                WHEN delivery_delay_days > 0 THEN 'Delayed'
                ELSE 'On time or early'
            END AS delivery_status
        FROM delivered_orders
    )
    
    SELECT
        delivery_status,
        COUNT(*) AS order_count,
        ROUND(
            100.0 * COUNT(*) /
            (SELECT COUNT(*) FROM classified_orders),
            2
        ) AS order_percent
    FROM classified_orders
    GROUP BY delivery_status
    ORDER BY order_count DESC
    """,
    [orders_file.as_posix()]
).df()

delivery_status_metrics

,delivery_status,order_count,order_percent
0,On time or early,89936,93.23
1,Delayed,6534,6.77


## 3. Delivery performance

Among 96,470 delivered orders with valid delivery dates, 89,936 orders arrived on time or early, while 6,534 orders arrived after the estimated delivery date.

The overall delivery-delay rate was 6.77%. Although delayed orders represent a minority of completed orders, their effect on customer satisfaction may still be substantial.

In [7]:
delay_rating_comparison = duckdb.execute(
    """
    WITH delivered_orders AS (
        SELECT
            order_id,
            DATE_DIFF(
                'day',
                CAST(order_estimated_delivery_date AS DATE),
                CAST(order_delivered_customer_date AS DATE)
            ) AS delivery_delay_days,
            CASE
                WHEN DATE_DIFF(
                    'day',
                    CAST(order_estimated_delivery_date AS DATE),
                    CAST(order_delivered_customer_date AS DATE)
                ) > 0
                THEN 'Delayed'
                ELSE 'On time or early'
            END AS delivery_status
        FROM read_csv_auto(?)
        WHERE order_status = 'delivered'
          AND order_delivered_customer_date IS NOT NULL
          AND order_estimated_delivery_date IS NOT NULL
    ),
    
    reviews_by_order AS (
        SELECT
            order_id,
            AVG(review_score) AS review_score
        FROM read_csv_auto(?)
        GROUP BY order_id
    ),
    
    analysis_data AS (
        SELECT
            o.order_id,
            o.delivery_delay_days,
            o.delivery_status,
            r.review_score
        FROM delivered_orders AS o
        INNER JOIN reviews_by_order AS r
            ON o.order_id = r.order_id
    )
    
    SELECT
        delivery_status,
        COUNT(*) AS orders_with_reviews,
        ROUND(AVG(review_score), 2) AS average_review_score
    FROM analysis_data
    GROUP BY delivery_status
    ORDER BY average_review_score DESC
    """,
    [
        orders_file.as_posix(),
        reviews_file.as_posix()
    ]
).df()

delay_rating_comparison

,delivery_status,orders_with_reviews,average_review_score
0,On time or early,89443,4.29
1,Delayed,6381,2.27


In [8]:
import os

sql_file = project_root / "sql" / "01_business_metrics.sql"
sql_script = sql_file.read_text(encoding="utf-8")

original_directory = Path.cwd()
connection = duckdb.connect()

try:
    os.chdir(project_root)
    final_sql_result = connection.execute(sql_script).df()
finally:
    os.chdir(original_directory)
    connection.close()

final_sql_result

,delivery_status,orders_with_reviews,average_review_score
0,On time or early,89443,4.29
1,Delayed,6381,2.27


## Preliminary findings

- The dataset contains 99,441 orders, of which 97.02% were marked as delivered.
- High ratings of four or five stars account for 77.07% of review records.
- Among delivered orders with valid delivery dates, 6.77% arrived later than the estimated date.
- On-time or early orders received an average review score of 4.29.
- Delayed orders received an average review score of 2.27.
- The preliminary review-score gap between the two delivery groups is approximately 2.02 points.

These results indicate a strong negative association between delivery delays and customer satisfaction. However, the findings remain preliminary until duplicate reviews, date consistency, outliers and other relevant factors have been examined.